In [0]:
%pip install pytrends urllib3==1.26.18
dbutils.library.restartPython()

In [0]:
from pytrends.request import TrendReq
import pandas as pd
from pyspark.sql.functions import col, current_timestamp

spark.sql("USE CATALOG scottish_housing_ws")
spark.sql("USE SCHEMA bronze")

In [0]:
keywords = [
    "house prices Scotland",
    "mortgage edinburgh",
    "Rightmove Scotland",
    "ESPC Edinburgh",
    "property for sale Edinburgh"
]

pytrends = TrendReq(hl='en-GB', tz=0, timeout=(10, 25), retries=3, backoff_factor=0.5)

pytrends.build_payload(
    kw_list=keywords,
    timeframe='today 5-y',
    geo='GB-SCT'
)

df_raw = pytrends.interest_over_time()

print(df_raw.shape)
print(df_raw.dtypes)
df_raw.head(10)

In [0]:
df_raw = df_raw.reset_index()
df_raw = df_raw.rename(columns={
    "date": "week_start_date",
    "house prices Scotland": "house_prices_scotland",
    "mortgage edinburgh": "mortgage_edinburgh",
    "Rightmove Scotland": "rightmove_scotland",
    "ESPC Edinburgh": "espc_edinburgh",
    "property for sale Edinburgh": "property_for_sale_edinburgh",
    "isPartial": "is_partial"
})
print(df_raw.columns.tolist())
print(df_raw.dtypes)
df_raw.head(5)

In [0]:
df_bronze = spark.createDataFrame(df_raw)
df_bronze.printSchema()
df_bronze.show(10, truncate=False)

In [0]:
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("scottish_housing_ws.bronze.google_trends")
)

In [0]:
# Source: Google Trends via pytrends (unofficial interface)
# Keywords: house prices Scotland, buy house Edinburgh, sell house Scotland,
#           ESPC Edinburgh, estate agent edinburgh
# Geography: GB-SCT (Scotland)
# Timeframe: today 5-y (approx 5 years of weekly data)
# Known limitation: single request window only -- multi-request stitching produces
# incomparable series due to per-request 0-100 rescaling.
# is_partial=True flags the most recent incomplete week.